##DD-167

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativeactive.dparty
ADD COLUMN FirstMotherNameGR string AFTER LastFatherNameENG;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativearchive.dparty
ADD COLUMN FirstMotherNameGR string AFTER LastFatherNameENG;
""")

In [0]:

spark.sql(f"""
ALTER TABLE `p-cdh`.primaryactive.party
ADD COLUMN FirstMotherNameGR string AFTER LastFatherNameENG;
""")

##DWH-4065

In [0]:
spark.sql("""
UPDATE `p-cdh`.primaryactive.portfolio
SET IsBankingFlag=false
WHERE portfolioname LIKE '%DOMUS%' 
   OR portfolioname LIKE '%RHODIUM%'
""")

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



##DOMUS-105

In [0]:
env = 'p'

In [0]:
# Drop derivativearchive.DExposure_backup & derivativeactive.DExposure_backup that already exist in prod
spark.sql( f"""
DROP TABLE IF EXISTS `{env}-cdh`.derivativearchive.DExposure_backup;
""")

spark.sql( f"""
DROP TABLE IF EXISTS `{env}-cdh`.derivativeactive.DExposure_backup;
""")

In [0]:
# Rename derivativearchive.DExposure & derivativeactive.DExposure for backup
spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativearchive.DExposure RENAME TO `{env}-cdh`.derivativearchive.DExposure_backup;
""")

spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativeactive.DExposure RENAME TO `{env}-cdh`.derivativeactive.DExposure_backup;
""")

In [0]:
# Change clustering columns to remove IsActiveFlagInt
spark.sql(f"""
ALTER TABLE `{env}-cdh`.derivativeactive.DExposure_backup
CLUSTER BY NONE;
""")

spark.sql(f"""
ALTER TABLE `{env}-cdh`.derivativearchive.DExposure_backup
CLUSTER BY NONE;
""")

In [0]:
# Drop column DExposure.IsActiveFlagInt
spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativearchive.DExposure_backup
DROP COLUMN IsActiveFlagInt;
""")

spark.sql( f"""
ALTER TABLE `{env}-cdh`.derivativeactive.DExposure_backup
DROP COLUMN IsActiveFlagInt;
""")

In [0]:
# Get all column names from the existing table schema
columns = spark.table(f"`{env}-cdh`.derivativeactive.DExposure_backup").schema.fields

# Convert columns to a name + datatype format
columns_formatted = [f"{col.name} {col.dataType.simpleString().upper()}" for col in columns]

# Ensure IsActiveFlagInt appears immediately after IsActiveFlag
reordered_columns = []
for col in columns:
    reordered_columns.append(f"{col.name} {col.dataType.simpleString().upper()}")
    if col.name == "IsActiveFlag":
        # Insert the new generated column right after IsActiveFlag
        reordered_columns.append("IsActiveFlagInt SMALLINT GENERATED ALWAYS AS (CAST(IsActiveFlag AS SMALLINT))")

# Build the CREATE TABLE SQL dynamically
create_table_sql = f"""
CREATE TABLE `{env}-cdh`.derivativeactive.DExposure (
    {', '.join(reordered_columns)}
)
USING DELTA
CLUSTER BY (BusinessDate, ExposureKey, IsActiveFlagInt)
TBLPROPERTIES (
    'delta.logRetentionDuration' = 'interval 5 weeks',
    'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7',
    'delta.columnMapping.mode' = 'name',
    'delta.dataSkippingStatsColumns' = 'BusinessDate, ExposureKey, IsActiveFlagInt'
)
"""

# Execute the query to create the table
spark.sql(create_table_sql)

In [0]:
# Get column names
columns = [col.name for col in spark.table(f"`{env}-cdh`.derivativeactive.DExposure").schema.fields if col.name != "IsActiveFlagInt"]

# Convert list to a string for SQL query
columns_str = ", ".join(columns)

# print(columns_str)

# Modify the query to explicitly list the columns
insert_sql = f"""
INSERT INTO `{env}-cdh`.derivativeactive.DExposure ({columns_str})
SELECT {columns_str}
FROM `{env}-cdh`.derivativeactive.DExposure_backup;
"""

# Execute the query
spark.sql(insert_sql)

In [0]:
# Get all column names from the existing table schema
columns = spark.table(f"`{env}-cdh`.derivativearchive.DExposure_backup").schema.fields

# Convert columns to a name + datatype format
columns_formatted = [f"{col.name} {col.dataType.simpleString().upper()}" for col in columns]

# Ensure IsActiveFlagInt appears immediately after IsActiveFlag
reordered_columns = []
for col in columns:
    reordered_columns.append(f"{col.name} {col.dataType.simpleString().upper()}")
    if col.name == "IsActiveFlag":
        # Insert the new generated column right after IsActiveFlag
        reordered_columns.append("IsActiveFlagInt SMALLINT GENERATED ALWAYS AS (CAST(IsActiveFlag AS SMALLINT))")

# Build the CREATE TABLE SQL dynamically
create_table_sql = f"""
CREATE TABLE `{env}-cdh`.derivativearchive.DExposure (
    {', '.join(reordered_columns)}
)
USING DELTA
CLUSTER BY (BusinessDate, ExposureKey, IsActiveFlagInt)
TBLPROPERTIES (
    'delta.logRetentionDuration' = 'interval 5 weeks',
    'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7',
    'delta.columnMapping.mode' = 'name',
    'delta.dataSkippingStatsColumns' = 'BusinessDate, ExposureKey, IsActiveFlagInt'
)
"""

# Execute the query to create the table
spark.sql(create_table_sql)

In [0]:
# Get column names
columns = [col.name for col in spark.table(f"`{env}-cdh`.derivativearchive.DExposure").schema.fields if col.name != "IsActiveFlagInt"]

# Convert list to a string for SQL query
columns_str = ", ".join(columns)

# print(columns_str)

# Modify the query to explicitly list the columns
insert_sql = f"""
INSERT INTO `{env}-cdh`.derivativearchive.DExposure ({columns_str})
SELECT {columns_str}
FROM `{env}-cdh`.derivativearchive.DExposure_backup;
"""

# Execute the query
spark.sql(insert_sql)